In [0]:
from pyspark import pipelines as dp
from trip_transforms import classify_trip_quality


@dp.temporary_view(name="yellow_trip_quality")
def yellow_trip_quality():
    return classify_trip_quality(
        spark.readStream.table("bronze.yellow_trips_raw"),
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
    )


@dp.temporary_view(name="green_trip_quality")
def green_trip_quality():
    return classify_trip_quality(
        spark.readStream.table("bronze.green_trips_raw"),
        "lpep_pickup_datetime",
        "lpep_dropoff_datetime",
    )


@dp.table(
    name="silver.yellow_trips_quarantine",
    comment="Rejected Yellow Taxi trips with source lineage and reason codes",
    table_properties={"delta.feature.timestampNtz": "supported"}
)
def yellow_trips_quarantine():
    return (
        spark.readStream.table("yellow_trip_quality")
        .filter("size(rejection_reasons) > 0")
    )


@dp.table(
    name="silver.green_trips_quarantine",
    comment="Rejected Green Taxi trips with source lineage and reason codes",
    table_properties={"delta.feature.timestampNtz": "supported"}
)
def green_trips_quarantine():
    return (
        spark.readStream.table("green_trip_quality")
        .filter("size(rejection_reasons) > 0")
    )


@dp.table(
    name="silver.yellow_trips",
    comment="Validated Yellow Taxi trips",
    table_properties={"delta.feature.timestampNtz": "supported"}
)
@dp.expect_or_drop("valid_yellow_trip", "size(rejection_reasons) = 0")
def yellow_trips():
    return spark.readStream.table("yellow_trip_quality")


@dp.table(
    name="silver.green_trips",
    comment="Validated Green Taxi trips",
    table_properties={"delta.feature.timestampNtz": "supported"}
)
@dp.expect_or_drop("valid_green_trip", "size(rejection_reasons) = 0")
def green_trips():
    return (
        spark.readStream
        .table("green_trip_quality")
        .drop("ehail_fee")
    )
